# Multi-Constellation Mosaic Visualization — Buenos Aires

This notebook mosaics the **extracted data** from four satellite constellations (GOES, MODIS, VIIRS, Sentinel-3) over the **entire Buenos Aires province** and renders them side-by-side for a direct visual comparison.

> **WOW effect:** See the same geographic region through four different sensors, resolutions, and spectral bands — all aligned to a unified mosaic.

## Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from aer.eoids import mosaic_eoids_tiles

## Configuration

Define the constellation directories and visual styling.

In [ ]:
# --- Configuration ---
BASE_DIR = Path(".")

CONSTELLATIONS = [
    {
        "label": "GOES-19 ABI",
        "dir": "extract_buenos_aires_goes",
        "cmap": "viridis",
        "band": "C07",
    },
    {
        "label": "MODIS Terra",
        "dir": "extract_buenos_aires_modis",
        "cmap": "plasma",
        "band": "31",
    },
    {
        "label": "VIIRS NOAA-21",
        "dir": "extract_buenos_aires_viirs",
        "cmap": "magma",
        "band": "I04",
    },
    {
        "label": "Sentinel-3 OLCI",
        "dir": "extract_buenos_aires_sentinel3",
        "cmap": "cividis",
        "band": "Oa04",
    },
]

TARGET_CRS = "EPSG:4326"

## Build mosaics

For each constellation, load all extracted tiles and merge them into a single mosaic covering the whole AOI.

In [ ]:
mosaics = {}

for cfg in CONSTELLATIONS:
    root = BASE_DIR / cfg["dir"]
    if not root.exists():
        print(f"Skipping {cfg['label']} — directory not found")
        continue

    print(f"Building mosaic for {cfg['label']} ...", flush=True)
    try:
        mosaic, transform, crs = mosaic_eoids_tiles(
            root,
            target_crs=TARGET_CRS,
        )
        mosaics[cfg["label"]] = {
            "data": mosaic,
            "transform": transform,
            "crs": crs,
            "cfg": cfg,
        }
        print(f"  -> shape {mosaic.shape}, CRS {crs}")
    except FileNotFoundError as e:
        print(f"  -> No tiles found: {e}")
    except Exception as e:
        print(f"  -> Error: {e}")

print(f"\nSuccessfully built {len(mosaics)} mosaics")

## Render side-by-side mosaic comparison

Display the full-area mosaic from each constellation in a unified figure.

Invalid values (nodata, zero-fill, and the VIIRS uint16 sentinel 65535) are masked so they do not distort the color scale.

In [ ]:
def mask_invalid(data, is_viirs=False):
    """Mask NaN, zero-fill, and VIIRS uint16 sentinel as NaN."""
    data = data.astype(float)
    invalid = np.isnan(data) | (data == 0)
    if is_viirs:
        invalid |= data == 65535
    data[invalid] = np.nan
    return data


def robust_vmin_vmax(data, lower=1, upper=99):
    """Compute percentile stretch excluding NaN."""
    valid = data[~np.isnan(data)]
    if len(valid) == 0:
        return 0, 1
    vmin, vmax = np.percentile(valid, [lower, upper])
    if vmin == vmax:
        vmin, vmax = valid.min(), valid.max()
    return vmin, vmax


if not mosaics:
    print("No mosaics to display. Run the extraction notebooks first!")
else:
    n = len(mosaics)
    cols = 2
    rows = (n + 1) // 2

    fig, axes = plt.subplots(rows, cols, figsize=(7 * cols, 6 * rows), squeeze=False)
    fig.suptitle(
        "Buenos Aires Province — Multi-Constellation Mosaic WOW Effect",
        fontsize=18,
        fontweight="bold",
        y=1.02,
    )

    for ax, (label, mosaic_info) in zip(axes.flat, mosaics.items()):
        raw = mosaic_info["data"]
        cfg = mosaic_info["cfg"]

        # Handle (bands, H, W) or (H, W) shapes
        if raw.ndim == 3:
            band_data = raw[0]
        else:
            band_data = raw

        is_viirs = cfg["band"] == "I04"
        band_data = mask_invalid(band_data, is_viirs=is_viirs)
        vmin, vmax = robust_vmin_vmax(band_data)

        im = ax.imshow(
            band_data,
            cmap=cfg["cmap"],
            vmin=vmin,
            vmax=vmax,
            interpolation="nearest",
        )
        ax.set_title(
            f"{label}\nBand {cfg['band']} | {band_data.shape}",
            fontsize=13,
            fontweight="bold",
        )
        ax.set_xlabel("Column")
        ax.set_ylabel("Row")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    # Hide any unused subplots
    for ax in axes.flat[n:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

## (Optional) Save the figure to disk

Uncomment the cell below to write the composite figure as a high-resolution PNG.

In [ ]:
# fig.savefig("multi_constellation_mosaic.png", dpi=200, bbox_inches="tight")
# print("Saved multi_constellation_mosaic.png")